# API-Sports.io NFL Weekly Stats Ingestion

Pull weekly NFL player statistics from API-Sports.io NFL API and land in bronze/silver Delta tables.

**Provider:** API-Sports.io (https://api-sports.io)  
**API Base:** `v1.american-football.api-sports.io`  
**Free Tier:** 100 requests/day  
**Authentication:** `x-apisports-key` header

In [0]:
import requests
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# API-Sports.io Configuration (NOT SportsData.io!)
API_KEY = "ba394c1e6cd4f63490c23b32430d1f3c"
BASE_URL = "https://v1.american-football.api-sports.io"  # NFL API endpoint

# Configure week and season - trying Week 1 first to test
WEEK = 1  # Testing with Week 1 which should have data
SEASON = 2024

print(f"📅 Fetching stats for Week {WEEK}, Season {SEASON} from API-Sports.io NFL")
print(f"   Base URL: {BASE_URL}")
print(f"   Free tier: 100 requests/day limit")

In [0]:
# API-Sports.io NFL - Fetch and parse player statistics
import requests
import json

# Use working authentication method
headers = {"x-apisports-key": API_KEY}

week_name = f"Week {WEEK}"

print(f"Fetching NFL data from API-Sports.io...")
print(f"Season: {SEASON}, Target Week: {week_name}\n")

# Fetch ALL games for the season
print(f"Step 1: Fetching ALL games for season {SEASON}...")
all_games_response = requests.get(
    f"{BASE_URL}/games",
    headers=headers,
    params={"league": "1", "season": str(SEASON)},
    timeout=30
)

if all_games_response.status_code == 200:
    all_games_data = all_games_response.json()
    all_games = all_games_data.get('response', [])
    print(f"✓ Found {len(all_games)} total games")
    
    # Filter for specific week
    week_games = [g for g in all_games if g.get('game', {}).get('week') == week_name]
    print(f"✓ Filtered to {len(week_games)} games for {week_name}")
    
    if week_games:
        # Extract game IDs
        game_ids = [g.get('game', {}).get('id') for g in week_games if g.get('game', {}).get('id')]
        print(f"✓ Found {len(game_ids)} game IDs\n")
        
        # Fetch player stats for all games
        print(f"Step 2: Fetching player stats for all {len(game_ids)} games...")
        print(f"   (Using {len(game_ids)} API requests)\n")
        
        all_player_records = []
        
        for idx, game_id in enumerate(game_ids, 1):
            game_stats_response = requests.get(
                f"{BASE_URL}/games/statistics/players",
                headers=headers,
                params={"id": str(game_id)},
                timeout=30
            )
            
            if game_stats_response.status_code == 200:
                game_stats = game_stats_response.json().get('response', [])
                
                # Parse nested structure: team → groups → players
                players_count = 0
                for team_data in game_stats:
                    team_name = team_data.get('team', {}).get('name')
                    groups = team_data.get('groups', [])
                    
                    # Loop through stat groups (Passing, Rushing, Receiving, etc.)
                    for group in groups:
                        group_name = group.get('name')
                        players = group.get('players', [])
                        
                        # Loop through players in this group
                        for player_entry in players:
                            player_info = player_entry.get('player', {})
                            player_id = player_info.get('id')
                            player_name = player_info.get('name')
                            
                            # Convert statistics array to dictionary
                            stats_array = player_entry.get('statistics', [])
                            stats_dict = {}
                            for stat in stats_array:
                                stat_name = stat.get('name', '').lower().replace(' ', '_')
                                stat_value = stat.get('value')
                                stats_dict[stat_name] = stat_value
                            
                            # Store player record with group context
                            all_player_records.append({
                                'player_id': player_id,
                                'player_name': player_name,
                                'team': team_name,
                                'stat_group': group_name,  # Passing, Rushing, Receiving, etc.
                                'statistics': stats_dict,
                                'game_id': game_id
                            })
                            players_count += 1
                
                print(f"   Game {idx}/{len(game_ids)}: +{players_count} player records")
            else:
                print(f"   Game {idx}/{len(game_ids)}: Failed ({game_stats_response.status_code})")
        
        print(f"\n✓ Total player records: {len(all_player_records)}\n")
        
        if all_player_records:
            # Show sample parsed data
            print("Sample parsed player record:")
            print(json.dumps(all_player_records[0], indent=2))
            print("\n...\n")
            
            # Convert to DataFrame format
            rows = []
            for record in all_player_records:
                # API-Sports.io doesn't provide fantasy points directly
                # Set to 0 for now (can calculate later based on scoring rules)
                fantasy_points = 0.0
                
                rows.append(
                    Row(
                        player_id=str(record['player_id']) if record['player_id'] else '',
                        week=WEEK,
                        season=SEASON,
                        fantasy_points=float(fantasy_points),
                        stats=json.dumps({
                            'player_name': record['player_name'],
                            'team': record['team'],
                            'stat_group': record['stat_group'],
                            'game_id': record['game_id'],
                            'statistics': record['statistics']
                        }),
                        source='api_sports'
                    )
                )
            
            stats_df = spark.createDataFrame(rows)
            
            # Remove duplicates (players may appear in multiple stat groups)
            # Keep all records since different stat groups provide different stats
            player_count = stats_df.select('player_id').distinct().count()
            
            print(f"✓ Created DataFrame with {stats_df.count()} total records")
            print(f"   ({player_count} unique players across all stat groups)\n")
            
            display(stats_df.limit(30))
            
            print(f"\n" + "="*60)
            print(f"\n📊 Summary:")
            print(f"   Games processed: {len(game_ids)}")
            print(f"   Total player-stat records: {stats_df.count()}")
            print(f"   Unique players: {player_count}")
            print(f"\n⚠️  Rate Limit: Used ~{1 + len(game_ids)} requests, ~{100 - 1 - len(game_ids)} remaining today")
        else:
            print("⚠️  No player records extracted")
    else:
        print(f"\n⚠️  No games found for {week_name}")
        # Show available weeks
        weeks = sorted(set([g.get('game', {}).get('week') for g in all_games if g.get('game', {}).get('week')]))
        print(f"   Available weeks: {weeks}")
else:
    print(f"✗ Failed: {all_games_response.status_code}")
    print(f"   {all_games_response.text[:300]}")

In [0]:
# Write to bronze table using MERGE
bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())

# Create temp view for merge
bronze_df.createOrReplaceTempView("api_sports_bronze_updates")

# Perform MERGE operation
spark.sql("""
  MERGE INTO main.fantasai.bronze_weekly_stats AS target
  USING api_sports_bronze_updates AS source
  ON target.player_id = source.player_id 
    AND target.week = source.week 
    AND target.season = source.season
  WHEN MATCHED THEN
    UPDATE SET
      target.fantasy_points = source.fantasy_points,
      target.stats = source.stats,
      target.ingested_at = source.ingested_at
  WHEN NOT MATCHED THEN
    INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
""")

print(f"✓ Merged {bronze_df.count()} records from API-Sports.io into bronze_weekly_stats")

In [0]:
# Transform for silver
silver_df = (
    bronze_df
    .select(
        F.col("player_id").cast("string"),
        F.col("week").cast("int"),
        F.col("season").cast("int"),
        F.col("fantasy_points").cast("double"),
        F.col("stats").cast("string"),
        F.col("ingested_at"),
    )
    .dropDuplicates(["player_id", "week", "season"])
)

# Create temp view for merge
silver_df.createOrReplaceTempView("api_sports_silver_updates")

# Perform MERGE operation
spark.sql("""
  MERGE INTO main.fantasai.silver_weekly_stats AS target
  USING api_sports_silver_updates AS source
  ON target.player_id = source.player_id 
    AND target.week = source.week 
    AND target.season = source.season
  WHEN MATCHED THEN
    UPDATE SET
      target.fantasy_points = source.fantasy_points,
      target.stats = source.stats,
      target.ingested_at = source.ingested_at
  WHEN NOT MATCHED THEN
    INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
""")

print(f"✓ Merged {silver_df.count()} records from API-Sports.io into silver_weekly_stats")